#### Dataset

In [ ]:
import pandas as pd
# from ydata_profiling import ProfileReport
%matplotlib inline

c:\Users\drkfa\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Cargar todos los datasets
customers = pd.read_csv("data/olist_customers_dataset.csv")
geolocation = pd.read_csv("data/olist_geolocation_dataset.csv")
order_items = pd.read_csv("data/olist_order_items_dataset.csv")
payments = pd.read_csv("data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("data/olist_order_reviews_dataset.csv")
orders = pd.read_csv("data/olist_orders_dataset.csv")
products = pd.read_csv("data/olist_products_dataset.csv")
sellers = pd.read_csv("data/olist_sellers_dataset.csv")
categories = pd.read_csv("data/product_category_name_translation.csv")


In [4]:
def safe_merge(df_left, df_right, on, how='left', validate=None, drop_duplicate_key=True, name="merge"):
    print(f"\n {name.upper()} — Uniendo por '{on}'")

    # Validar unicidad en df_right si se espera many_to_one o one_to_one
    if validate in ['many_to_one', 'one_to_one']:
        if not df_right[on].is_unique:
            print(f" La clave '{on}' en df_right tiene duplicados.")
            if drop_duplicate_key:
                print(f" Eliminando duplicados en '{on}' de df_right...")
                df_right = df_right.drop_duplicates(subset=on)
            else:
                raise ValueError(f"Clave '{on}' no es única en df_right y no se permite continuar.")

    # Filas antes
    before_rows = df_left.shape[0]

    # Merge
    df_merged = df_left.merge(df_right, on=on, how=how, validate=validate)

    # Filas después
    after_rows = df_merged.shape[0]

    print(f" Filas antes: {before_rows} → después: {after_rows}")
    if after_rows > before_rows:
        print(" Se crearon filas duplicadas. Revisar claves o aplicar agrupamiento.")
    else:
        print(" No se crearon duplicados.")

    return df_merged


In [5]:
# Unir reviews con orders (por order_id)
#df = reviews.merge(orders, on="order_id", how="left")

df = safe_merge(
    df_left=reviews,
    df_right=orders,
    on="order_id",
    how="left",
    validate='many_to_one',  # cada order_id aparece una vez en orders
    name='reviews + orders'
)



 REVIEWS + ORDERS — Uniendo por 'order_id'
 Filas antes: 99224 → después: 99224
 No se crearon duplicados.


In [6]:
# Unir con payments (por order_id)
# Agrupar y resumir payments
payments_summary = payments.groupby("order_id").agg({
    "payment_value": "sum",
    "payment_installments": "mean",
    "payment_type": lambda x: x.mode()[0] if not x.mode().empty else "unknown"
}).reset_index()

df = safe_merge(
    df_left=df,
    df_right=payments_summary,
    on="order_id",
    how="left",
    validate='many_to_one',  
    name='payments_summary'
)


 PAYMENTS_SUMMARY — Uniendo por 'order_id'
 Filas antes: 99224 → después: 99224
 No se crearon duplicados.


In [7]:
# Unir con order_items (por order_id)
# Agrupar y resumir order_items
order_items_summary = order_items.groupby("order_id").agg({
    "price": "sum",
    "freight_value": "sum",
    "product_id": lambda x: x.mode()[0] if not x.mode().empty else None,
    "seller_id": lambda x: x.mode()[0] if not x.mode().empty else None,
    "order_item_id": "count"
}).rename(columns={"order_item_id": "num_items"}).reset_index()

df = safe_merge(
    df_left=df,
    df_right=order_items_summary,
    on="order_id",
    how="left",
    validate='many_to_one',  
    name='order_items_summary'
)


 ORDER_ITEMS_SUMMARY — Uniendo por 'order_id'
 Filas antes: 99224 → después: 99224
 No se crearon duplicados.


In [8]:
# Unir con customers (por customer_id)
df = safe_merge(
    df_left=df,
    df_right=customers,
    on="customer_id",
    how="left",
    validate='many_to_one',  
    name='customers'
)


 CUSTOMERS — Uniendo por 'customer_id'
 Filas antes: 99224 → después: 99224
 No se crearon duplicados.


In [9]:
# Asegurarse de que no haya duplicado antes del merge
if 'product_id' in df.columns:
    df.drop(columns='product_id', inplace=True)

#  merge con product_id desde order_items
df = safe_merge(
    df_left=df,
    df_right=order_items[['order_id', 'product_id']],
    on='order_id',
    how='left',
    validate='many_to_one',  
    name='order_items (product_id)'
)

#  merge con products
df = safe_merge(
    df_left=df,
    df_right=products,
    on='product_id',
    how='left',
    validate='many_to_one',  
    name='products'
)


 ORDER_ITEMS (PRODUCT_ID) — Uniendo por 'order_id'
 La clave 'order_id' en df_right tiene duplicados.
 Eliminando duplicados en 'order_id' de df_right...
 Filas antes: 99224 → después: 99224
 No se crearon duplicados.

 PRODUCTS — Uniendo por 'product_id'
 Filas antes: 99224 → después: 99224
 No se crearon duplicados.


In [10]:
# Asegurarse de que no haya duplicado antes del merge
if 'seller_id' in df.columns:
    df.drop(columns='seller_id', inplace=True)

# obtener seller_id más frecuente por order_id
seller_ids = order_items.groupby('order_id')['seller_id'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else None
).reset_index()

# merge con seller_id
df = safe_merge(
    df_left=df,
    df_right=seller_ids,
    on='order_id',
    how='left',
    validate='many_to_one',  
    name='seller_ids'
)

# merge con sellers
df = safe_merge(
    df_left=df,
    df_right=sellers,
    on='seller_id',
    how='left',
    validate='many_to_one',  
    name='sellers'
)


 SELLER_IDS — Uniendo por 'order_id'
 Filas antes: 99224 → después: 99224
 No se crearon duplicados.

 SELLERS — Uniendo por 'seller_id'
 Filas antes: 99224 → después: 99224
 No se crearon duplicados.


In [11]:
# Unir con traducción de categoría (por product_category_name)
df = safe_merge(
    df_left=df,
    df_right=categories,
    on="product_category_name",
    how="left",
    validate='many_to_one',  
    name='categorías traducidas'
)



 CATEGORÍAS TRADUCIDAS — Uniendo por 'product_category_name'
 Filas antes: 99224 → después: 99224
 No se crearon duplicados.


In [12]:

# Crear el reporte
#profile = ProfileReport(df, title="Reporte del Dataset Unificado", explorative=True)

# Guardarlo como archivo HTML
#profile.to_file("reporte_dataset_unificado.html")


### Limpieza del dataset

In [13]:
# Renombrar las columnas a conservar
df.rename(columns={
    'freight_value_x': 'freight_value',
    'price_x': 'price',
    'num_items_x': 'num_items',
    'product_category_name_english_x': 'product_category_name_english'
}, inplace=True)

In [14]:
# Eliminar columnas con sufijo _y 
df.drop(columns=[
    'freight_value_y',
    'price_y',
    'num_items_y',
    'product_category_name_english_y'
], errors='ignore', inplace=True)


In [15]:
# Eliminar cualquier versión de product_id y seller_id (x, y)
df.drop(columns=[
    'product_id_x', 'product_id_y',
    'seller_id_x', 'seller_id_y'
], errors='ignore', inplace=True)

In [16]:
# Corregir errores de escritura en nombres de columnas
df.columns = [col.replace('lenght', 'length') for col in df.columns]

In [17]:
# Asegurar que sean fechas
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])

# Variables derivadas útiles
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['is_late'] = (df['order_delivered_customer_date'] > df['order_estimated_delivery_date']).astype(int)
df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek

In [18]:
df.drop(columns=[
    'order_id', 'customer_id', 'seller_id', 'product_id', 'review_id',
    'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
    'order_delivered_customer_date', 'order_estimated_delivery_date',
    'review_creation_date', 'review_answer_timestamp', 'price', 'product_width_cm', 'product_height_cm', 'product_length_cm'
], errors='ignore', inplace=True)


In [19]:
df.head(20)

,review_score,review_comment_title,review_comment_message,order_status,payment_value,payment_installments,payment_type,freight_value,num_items,customer_unique_id,...,product_description_length,product_photos_qty,product_weight_g,seller_zip_code_prefix,seller_city,seller_state,product_category_name_english,delivery_days,is_late,purchase_dayofweek
0,4,NaN,NaN,delivered,397.26,8.0,credit_card,27.26,2.0,68a5590b9926689be4e10f4ae2db21a8,...,858.0,1.0,1300.0,14600.0,sao joaquim da barra,SP,sports_leisure,6.0,0,3
1,5,NaN,NaN,delivered,88.09,1.0,credit_card,8.30,1.0,64190b91b656ab8f37eb89b93dc84584,...,493.0,1.0,245.0,12233.0,sao jose dos campos,SP,computers_accessories,9.0,0,2
2,5,NaN,NaN,delivered,194.12,1.0,credit_card,45.12,1.0,1d47144362c14e94ccdd213e8ec277d5,...,1893.0,1.0,6550.0,37175.0,ilicinea,MG,computers_accessories,13.0,0,5
3,5,NaN,Recebi bem antes do prazo estipulado.,delivered,222.84,1.0,credit_card,42.85,1.0,c8cf6cb6b838dc7a33ed199b825e8616,...,2188.0,2.0,7650.0,37175.0,ilicinea,MG,garden_tools,10.0,0,6
4,5,NaN,Parabéns lojas lannister adorei comprar pela I...,delivered,1333.25,10.0,credit_card,134.25,1.0,d16000272660a1fef81482ad75ba572a,...,562.0,5.0,9850.0,81730.0,curitiba,PR,sports_leisure,18.0,0,5
5,1,NaN,NaN,delivered,462.70,1.0,credit_card,44.00,4.0,bda84be75dfc9588ae63cfe827080b9b,...,246.0,2.0,950.0,13405.0,piracicaba,SP,bed_bath_table,5.0,0,4
6,5,NaN,NaN,delivered,58.75,2.0,credit_card,11.85,1.0,dcd4940b3f96a3e2b73d8f73387230cf,...,133.0,1.0,600.0,14940.0,ibitinga,SP,bed_bath_table,14.0,0,4
7,5,NaN,NaN,delivered,198.96,3.0,credit_card,59.06,1.0,c7b4fb0959a97e7033ff9bef3b1e2ba9,...,1213.0,9.0,8450.0,16304.0,penapolis,SP,toys,5.0,0,1
8,5,NaN,NaN,delivered,102.03,3.0,credit_card,12.13,1.0,8c89391790076834500661cc1e5d6860,...,176.0,1.0,1825.0,14940.0,ibitinga,SP,home_confort,8.0,0,0
9,4,recomendo,aparelho eficiente. no site a marca do aparelh...,delivered,613.25,8.0,credit_card,26.69,1.0,2bf6fd4ad93eb21b3d604481c48decbf,...,3839.0,4.0,1450.0,98803.0,santo angelo,RS,small_appliances,7.0,0,0


In [20]:
df.describe()

,review_score,payment_value,payment_installments,freight_value,num_items,customer_zip_code_prefix,product_name_length,product_description_length,product_photos_qty,product_weight_g,seller_zip_code_prefix,delivery_days,is_late,purchase_dayofweek
count,99224.000000,99223.000000,99223.000000,98465.000000,98465.000000,99224.000000,97051.000000,97051.000000,97051.000000,98450.000000,98465.000000,96359.000000,99224.000000,99224.000000
mean,4.086421,160.564127,2.915955,22.799370,1.141238,35157.108986,48.850357,793.172384,2.248210,2098.231752,24621.991926,12.058957,0.077612,2.757004
std,1.347579,220.316047,2.701399,21.594256,0.534980,29823.027753,9.996261,654.280548,1.746075,3761.133760,27692.162908,9.463076,0.267562,1.966358
min,1.000000,0.000000,0.000000,0.000000,1.000000,1003.000000,5.000000,4.000000,1.000000,0.000000,1001.000000,0.000000,0.000000,0.000000
25%,4.000000,61.880000,1.000000,13.840000,1.000000,11340.000000,42.000000,348.000000,1.000000,300.000000,6440.000000,6.000000,0.000000,1.000000
50%,5.000000,105.280000,2.000000,17.160000,1.000000,24415.000000,52.000000,606.000000,2.000000,700.000000,13568.000000,10.000000,0.000000,3.000000
75%,5.000000,176.710000,4.000000,24.010000,1.000000,59022.000000,57.000000,995.000000,3.000000,1800.000000,29156.000000,15.000000,0.000000,4.000000
max,5.000000,13664.080000,24.000000,1794.960000,21.000000,99990.000000,76.000000,3992.000000,20.000000,40425.000000,99730.000000,208.000000,1.000000,6.000000


In [21]:
df.isnull().sum().sort_values(ascending=False)

review_comment_title             87656
review_comment_message           58247
delivery_days                     2865
product_category_name_english     2194
product_photos_qty                2173
product_description_length        2173
product_name_length               2173
product_category_name             2173
product_weight_g                   774
seller_state                       759
seller_city                        759
freight_value                      759
seller_zip_code_prefix             759
num_items                          759
payment_type                         1
payment_value                        1
payment_installments                 1
order_status                         0
review_score                         0
customer_city                        0
customer_state                       0
customer_unique_id                   0
customer_zip_code_prefix             0
is_late                              0
purchase_dayofweek                   0
dtype: int64

In [22]:
# Eliminar columnas innecesarias
cols_to_drop = [
    'review_comment_title',
    'review_comment_message',
    'product_category_name',       
    'seller_zip_code_prefix', 
    'customer_city',
    'seller_city' ,   
    'customer_unique_id', 
    'customer_zip_code_prefix',
    'is_late'
    ]

df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [23]:
# Eliminar filas con valores nulos 
df = df[df['num_items'].notna()]

# Eliminar filas con delivery_days nulo 
df = df[df['delivery_days'].notna()]


In [24]:
# Imputar variables numéricas con mediana
num_median_impute = [
    'product_description_length',
    'product_name_length',
    'product_photos_qty',
    'product_weight_g',
    'freight_value'
]
for col in num_median_impute:
    df[col].fillna(df[col].median(), inplace=True)


C:\Users\drkfa\AppData\Local\Temp\ipykernel_14204\2423929854.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\drkfa\AppData\Local\Temp\ipykernel_14204\2423929854.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

In [25]:
# Imputar categóricas con 'missing'
cat_fill = [
    'product_category_name_english',
    'seller_state',
    'payment_type'
]
for col in cat_fill:
    df[col].fillna('unknown', inplace=True)

C:\Users\drkfa\AppData\Local\Temp\ipykernel_14204\2004608745.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna('unknown', inplace=True)


In [26]:
#  Imputar numérica con mediana
df['payment_installments'].fillna(df['payment_installments'].median(), inplace=True)
df['payment_value'].fillna(df['payment_value'].median(), inplace=True)

C:\Users\drkfa\AppData\Local\Temp\ipykernel_14204\2576313910.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['payment_installments'].fillna(df['payment_installments'].median(), inplace=True)
C:\Users\drkfa\AppData\Local\Temp\ipykernel_14204\2576313910.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting value

In [27]:
df.isnull().sum().sort_values(ascending=False)

review_score                     0
order_status                     0
payment_value                    0
payment_installments             0
payment_type                     0
freight_value                    0
num_items                        0
customer_state                   0
product_name_length              0
product_description_length       0
product_photos_qty               0
product_weight_g                 0
seller_state                     0
product_category_name_english    0
delivery_days                    0
purchase_dayofweek               0
dtype: int64

In [28]:
df.describe()


,review_score,payment_value,payment_installments,freight_value,num_items,product_name_length,product_description_length,product_photos_qty,product_weight_g,delivery_days,purchase_dayofweek
count,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000
mean,4.155554,159.439157,2.913514,22.757456,1.141689,48.926608,789.980075,2.245571,2092.420386,12.058957,2.757895
std,1.285108,217.157533,2.698149,21.495454,0.535353,9.914882,649.111755,1.733943,3747.920169,9.463076,1.966929
min,1.000000,9.590000,0.000000,0.000000,1.000000,5.000000,4.000000,1.000000,0.000000,0.000000,0.000000
25%,4.000000,61.800000,1.000000,13.840000,1.000000,43.000000,352.000000,1.000000,300.000000,6.000000,1.000000
50%,5.000000,105.130000,2.000000,17.160000,1.000000,52.000000,606.000000,2.000000,700.000000,10.000000,3.000000
75%,5.000000,176.090000,4.000000,23.990000,1.000000,57.000000,985.000000,3.000000,1800.000000,15.000000,4.000000
max,5.000000,13664.080000,24.000000,1794.960000,21.000000,76.000000,3992.000000,20.000000,40425.000000,208.000000,6.000000


In [29]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

# Variables categóricas con pocas categorías
categorical_onehot = ['order_status', 'payment_type']

# Variables categóricas con muchas categorías 
categorical_ordinal = ['customer_state', 'seller_state', 'product_category_name_english']

# Variables numéricas a escalar
numerical_to_scale = [
    'payment_value', 'payment_installments', 'freight_value',
    'product_weight_g',
    'delivery_days', 'product_description_length', 'product_name_length'
]

# ColumnTransformer con codificadores apropiados
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_to_scale),
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_onehot),
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_ordinal)
    ],
    remainder='passthrough'  
)



In [30]:
X = df.drop(columns='num_items')  # Features
y = df['num_items']               # Target


In [31]:
X.columns.difference(categorical_onehot + categorical_ordinal + numerical_to_scale)


Index(['product_photos_qty', 'purchase_dayofweek', 'review_score'], dtype='object')

In [32]:
X_transformed = preprocessor.fit_transform(X)


In [33]:
# Nombres de columnas escaladas
num_feature_names = preprocessor.named_transformers_['num'].get_feature_names_out(numerical_to_scale)

# Nombres de columnas categóricas codificadas
onehot_feature_names = preprocessor.named_transformers_['onehot'].get_feature_names_out(categorical_onehot)
ordinal_feature_names = categorical_ordinal  # ordinal encoder no transforma nombres

# Nombres de columnas pasadas sin transformar
passthrough_cols = [col for col in X.columns if col not in numerical_to_scale + categorical_onehot + categorical_ordinal]

# Unir todos los nombres
final_column_names = list(num_feature_names) + list(onehot_feature_names) + list(ordinal_feature_names) + passthrough_cols


In [34]:
import numpy as np

if not isinstance(X_transformed, np.ndarray):
    X_transformed = X_transformed.toarray()
    
X_df = pd.DataFrame(X_transformed, columns=final_column_names)
X_df.head()

,payment_value,payment_installments,freight_value,product_weight_g,delivery_days,product_description_length,product_name_length,order_status_delivered,payment_type_credit_card,payment_type_debit_card,payment_type_unknown,payment_type_voucher,customer_state,seller_state,product_category_name_english,review_score,product_photos_qty,purchase_dayofweek
0,1.095159,1.885186,0.209466,-0.211430,-0.640277,0.104790,-0.698611,1.0,1.0,0.0,0.0,0.0,25.0,21.0,65.0,4.0,1.0,3.0
1,-0.328561,-0.709199,-0.672585,-0.492921,-0.323254,-0.457520,-0.194316,1.0,1.0,0.0,0.0,0.0,25.0,21.0,15.0,5.0,1.0,2.0
2,0.159704,-0.709199,1.040344,1.189354,0.099444,1.699285,1.015992,1.0,1.0,0.0,0.0,0.0,4.0,7.0,15.0,5.0,1.0,5.0
3,0.291959,-0.709199,0.934739,1.482851,-0.217579,2.153754,-1.606342,1.0,1.0,0.0,0.0,0.0,23.0,7.0,42.0,5.0,2.0,6.0
4,5.405371,2.626438,5.186823,2.069847,0.627816,-0.351220,-0.093457,1.0,1.0,0.0,0.0,0.0,23.0,14.0,65.0,5.0,5.0,5.0


In [35]:
X_df.describe()

,payment_value,payment_installments,freight_value,product_weight_g,delivery_days,product_description_length,product_name_length,order_status_delivered,payment_type_credit_card,payment_type_debit_card,payment_type_unknown,payment_type_voucher,customer_state,seller_state,product_category_name_english,review_score,product_photos_qty,purchase_dayofweek
count,9.635900e+04,9.635900e+04,9.635900e+04,9.635900e+04,9.635900e+04,9.635900e+04,9.635900e+04,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000,96359.000000
mean,5.493564e-18,-1.541885e-16,-1.299283e-16,-3.391999e-18,-1.011701e-16,-9.667198e-17,2.501231e-16,0.999938,0.766208,0.015401,0.000010,0.019344,18.651792,18.416619,39.724136,4.155554,2.245571,2.757895
std,1.000005e+00,1.000005e+00,1.000005e+00,1.000005e+00,1.000005e+00,1.000005e+00,1.000005e+00,0.007891,0.423244,0.123141,0.003221,0.137733,7.082311,4.853460,22.912048,1.285108,1.733943,1.966929
min,-6.900517e-01,-1.079825e+00,-1.058716e+00,-5.582913e-01,-1.274323e+00,-1.210861e+00,-4.430394e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000
25%,-4.496260e-01,-7.091989e-01,-4.148552e-01,-4.782465e-01,-6.402768e-01,-6.747410e-01,-5.977518e-01,1.000000,1.000000,0.000000,0.000000,0.000000,12.000000,18.000000,15.000000,4.000000,1.000000,1.000000
50%,-2.500924e-01,-3.385726e-01,-2.604032e-01,-3.715201e-01,-2.175791e-01,-2.834351e-01,3.099793e-01,1.000000,1.000000,0.000000,0.000000,0.000000,22.000000,21.000000,43.000000,5.000000,2.000000,3.000000
75%,7.667673e-02,4.026802e-01,5.734007e-02,-7.802245e-02,3.107931e-01,3.004428e-01,8.142743e-01,1.000000,1.000000,0.000000,0.000000,0.000000,25.000000,21.000000,65.000000,5.000000,3.000000,4.000000
max,6.218854e+01,7.815208e+00,8.244589e+01,1.022775e+01,2.070596e+01,4.932951e+00,2.730595e+00,1.000000,1.000000,1.000000,1.000000,1.000000,26.000000,21.000000,71.000000,5.000000,20.000000,6.000000


In [36]:
df_full = X_df.copy()
df_full['num_items'] = y.reset_index(drop=True)


In [37]:
from scipy.stats import zscore

# Calcular Z-score sobre variables numéricas
z_scores = np.abs(zscore(df_full[numerical_to_scale]))
mask = (z_scores < 3).all(axis=1)

# Filtrar sin outliers
df_no_outliers = df_full[mask]

In [38]:
#df_full = df.drop_duplicates()
#df_no_outliers = df.drop_duplicates()

In [39]:
df_full.to_csv("dataset_full.csv", index=False)
df_no_outliers.to_csv("dataset_no_outliers.csv", index=False)
